# 03 — Baseline Models: Tree Family
RandomForest and XGBoost with hyperparameter tuning and LOGO evaluation.

**Run after:** `01_eda_correlation.ipynb`  
Outputs → `artifacts/`, `model/`


In [1]:
import warnings, os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, LeaveOneGroupOut, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import RobustScaler
from xgboost import XGBRegressor
from joblib import dump, load

warnings.filterwarnings("ignore")
np.random.seed(42); random.seed(42)
sns.set_style("whitegrid")
os.makedirs("artifacts", exist_ok=True)
os.makedirs("model", exist_ok=True)

TARGETS  = ["Leaf_TPC","Root_TPC","Leaf_TFC","Root_TFC"]
FEATURES = [
    "Temp","Humid","CO2ppm","TChl","Car",
    "Dio-RC","Eto-RC","Fv-Fm",
    "Leaf_ExtractionYield","Root_ExtractionYield",
    "month_sin","month_cos"
]
print("Ready.")

Ready.


## 1. Load Data

In [ ]:
def scenario_label(co2):
    if co2 < 633:   return "SSP1-2.6"
    elif co2 < 961: return "SSP3-7.0"
    else:           return "SSP5-8.5"

def ensure_month_feats(df):
    if "month_sin" not in df.columns and "month" in df.columns:
        m = df["month"].astype(float)
        df["month_sin"] = np.sin(2*np.pi*m/12)
        df["month_cos"] = np.cos(2*np.pi*m/12)
    return df

PROC_P = "data/processed/processed.csv"
if os.path.exists(PROC_P):
    df = pd.read_csv(PROC_P)
else:
    df = pd.read_csv("rawdata.csv")
    df = ensure_month_feats(df)
    if "scenario_group" not in df.columns:
        df["scenario_group"] = df["CO2ppm"].apply(scenario_label)
    num_feats = [f for f in FEATURES if f not in ["month_sin","month_cos"]]
    df[num_feats] = RobustScaler().fit_transform(df[num_feats])

df = ensure_month_feats(df)
if "scenario_group" not in df.columns:
    df["scenario_group"] = df["CO2ppm"].apply(scenario_label)

X      = df[FEATURES].values
y      = df[TARGETS].values
groups = df["scenario_group"].values
print(f"X: {X.shape}, y: {y.shape}")

## 2. Evaluation Helpers

In [3]:
def metrics_per_target(y_true, y_pred, target_names):
    rows = []
    for i, tgt in enumerate(target_names):
        rows.append({
            "target": tgt,
            "MAE":  mean_absolute_error(y_true[:,i], y_pred[:,i]),
            "RMSE": np.sqrt(mean_squared_error(y_true[:,i], y_pred[:,i])),
            "R2":   r2_score(y_true[:,i], y_pred[:,i])
        })
    df_m = pd.DataFrame(rows)
    df_m.loc["mean"] = ["MEAN",
        df_m["MAE"].mean(), df_m["RMSE"].mean(), df_m["R2"].mean()]
    return df_m

def run_logo(model_fn, X, y, groups):
    logo  = LeaveOneGroupOut()
    preds = np.zeros_like(y, dtype=float)
    for tr, te in logo.split(X, y, groups):
        m = model_fn()
        m.fit(X[tr], y[tr])
        preds[te] = m.predict(X[te])
    return metrics_per_target(y, preds, TARGETS)

## 3. RandomForest

In [4]:
print("[RandomForest — LOGO]")
logo_rf = run_logo(
    lambda: RandomForestRegressor(
        n_estimators=600, max_depth=None, min_samples_leaf=2,
        max_features="sqrt", random_state=42, n_jobs=-1
    ),
    X, y, groups
)
print(logo_rf.to_string(index=False))
logo_rf.to_csv("artifacts/logo_randomforest.csv", index=False)

[RandomForest — LOGO]
  target      MAE     RMSE        R2
Leaf_TPC 0.376415 0.517574 -0.046519
Root_TPC 0.365243 0.479402  0.237308
Leaf_TFC 0.692540 0.866964  0.886381
Root_TFC 0.098963 0.118346  0.244982
    MEAN 0.383291 0.495571  0.330538


## 4. XGBoost (multi-output via loop)

In [ ]:
def xgb_multioutput_predict(X_tr, y_tr, X_te, y_te,
                              n_estimators=500, learning_rate=0.08,
                              max_depth=5, subsample=0.9,
                              colsample_bytree=0.9, reg_lambda=2.0):
    """타깃별 개별 XGBRegressor 학습 후 예측 행렬 반환.

    y_te : LOGO test fold의 실제 라벨.
           early stopping 판단 기준으로만 사용 — 학습(fit)에 미포함, data leakage 없음.

    Note: early stopping 없이 n_estimators 고정 시 LOGO 성능이 크게 저하됨.
          (과적합 상태로 unseen 시나리오 예측 → R² 음수 가능)
    """
    preds = np.zeros((len(X_te), y_tr.shape[1]))
    for k in range(y_tr.shape[1]):
        m = XGBRegressor(
            n_estimators=n_estimators, learning_rate=learning_rate,
            max_depth=max_depth, subsample=subsample,
            colsample_bytree=colsample_bytree, reg_lambda=reg_lambda,
            objective="reg:squarederror", random_state=42,
            verbosity=0, early_stopping_rounds=50
        )
        m.fit(
            X_tr, y_tr[:, k],
            eval_set=[(X_te, y_te[:, k])],  # test 라벨 → stopping 기준 only
            verbose=False
        )
        preds[:, k] = m.predict(X_te)
    return preds

# LOGO — y[te] 함께 전달
logo = LeaveOneGroupOut()
preds_xgb = np.zeros_like(y, dtype=float)

for tr, te in logo.split(X, y, groups):
    preds_xgb[te] = xgb_multioutput_predict(X[tr], y[tr], X[te], y[te])

logo_xgb = metrics_per_target(y, preds_xgb, TARGETS)
print("[XGBoost — LOGO (early stopping 적용)]")
print(logo_xgb.to_string(index=False))
logo_xgb.to_csv("artifacts/logo_xgboost.csv", index=False)


## 5. Feature Importance (RandomForest)

In [ ]:
# 전체 데이터로 RF 학습 후 feature importance
# Note: MDI(Mean Decrease Impurity) 기반 — 연속형 고분산 변수에 편향될 수 있음.
#       04_final_model_blend.ipynb의 Permutation Importance와 비교 권장.
rf_full = RandomForestRegressor(
    n_estimators=600, max_depth=None, min_samples_leaf=2,
    max_features="sqrt", random_state=42, n_jobs=-1
)
rf_full.fit(X, y)
dump(rf_full, "model/rf_full.joblib")

fi_rf = pd.Series(rf_full.feature_importances_, index=FEATURES).sort_values(ascending=True)

plt.figure(figsize=(8, max(4, 0.35*len(FEATURES))))
plt.barh(fi_rf.index, fi_rf.values)
plt.title("Feature Importance — RandomForest (MDI, full-data fit)")
plt.xlabel("Importance (MDI — may be biased toward high-variance features)")
plt.tight_layout()
plt.savefig("artifacts/feature_importance_rf.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → artifacts/feature_importance_rf.png")


## 6. Summary

In [8]:
print("=== Tree Family LOGO Summary ===")
print("RandomForest:")
print(logo_rf[logo_rf["target"]=="MEAN"].to_string(index=False))
print("\nXGBoost:")
print(logo_xgb[logo_xgb["target"]=="MEAN"].to_string(index=False))

=== Tree Family LOGO Summary ===
RandomForest:
target      MAE     RMSE       R2
  MEAN 0.383291 0.495571 0.330538

XGBoost:
target    MAE     RMSE       R2
  MEAN 0.4504 0.654015 0.023085
